# Classification with PyTorch V2.1 (oracle) — Response, FT-Transformer-lite, k=5

**Version:** V2.1 (k-fold validation — novel architectures). **Role:** reference oracle for the V2.1
scratch twin. Same `FTTransformerLite` architecture and hypers as `classification_pytorch_V2_0.ipynb`
(per-feature tokenizer + CLS token + 2x PreNorm transformer blocks, weighted BCE, Adam+cosine+wd(1e-4),
100 epochs, best-val-F1 checkpoint), copied verbatim and wrapped in a 5-fold `StratifiedKFold` loop over
the full 2,236-row cleaned set instead of one fixed 80/20 split. The numeric grad-check runs once
(architecture-only, not data-dependent) rather than per fold. Weights → `./weights_V2_1/`.

In [1]:
import json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("weights_V2_1", exist_ok=True)

## 1. LOAD DATA (identical to V2.0)

In [2]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
print("cleaned:", df.shape, "| Response rate:", round(df["Response"].mean(), 4))

CLF_NUMERIC = ["AcceptedCmp5","AcceptedCmp3","MntWines","MntMeatProducts","NumCatalogPurchases",
    "Recency","Customer_Tenure_Days","MntGoldProds","Income","MntFruits","AcceptedCmp1",
    "AcceptedCmp2","NumWebPurchases","MntSweetProducts","MntFishProducts","NumStorePurchases","Age"]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
inc_idx = FEATURE_COLS.index("Income")
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {len(FEATURE_COLS)} dims")

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32)).view(-1, 1)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

cleaned: (2236, 28) | Response rate: 0.1494
features: 17 numeric + 5 dummies = 22 dims


## 2. MODEL: FT-Transformer-lite (identical to V2.0)

In [3]:
class TransformerBlock(nn.Module):
    def __init__(self, d=32, h=4, p=0.1):
        super().__init__()
        assert d % h == 0; self.h, self.dk = h, d // h
        self.ln1 = nn.LayerNorm(d, eps=1e-5); self.ln2 = nn.LayerNorm(d, eps=1e-5)
        self.wq = nn.Linear(d, d); self.wk = nn.Linear(d, d); self.wv = nn.Linear(d, d); self.wo = nn.Linear(d, d)
        self.fc1 = nn.Linear(d, 2*d); self.fc2 = nn.Linear(2*d, d)
        self.relu = nn.ReLU(); self.drop = nn.Dropout(p); self.attn_drop = nn.Dropout(p)
        for fc in (self.wq, self.wk, self.wv, self.wo, self.fc1, self.fc2):
            nn.init.kaiming_normal_(fc.weight, nonlinearity="relu"); nn.init.zeros_(fc.bias)
    def forward(self, t):
        B, T, D = t.shape
        h_in = self.ln1(t)
        Q = self.wq(h_in).view(B, T, self.h, self.dk).transpose(1, 2)
        K = self.wk(h_in).view(B, T, self.h, self.dk).transpose(1, 2)
        V = self.wv(h_in).view(B, T, self.h, self.dk).transpose(1, 2)
        s = (Q @ K.transpose(-2, -1)) / (self.dk ** 0.5)
        p = self.attn_drop(torch.softmax(s, dim=-1))
        o = (p @ V).transpose(1, 2).reshape(B, T, D)
        u = t + self.wo(o)
        v = u + self.fc2(self.drop(self.relu(self.fc1(self.ln2(u)))))
        return v

class FTTransformerLite(nn.Module):
    def __init__(self, in_dim=22, d=32, layers=2, heads=4, p=0.1):
        super().__init__()
        self.tok = nn.ModuleList([nn.Linear(1, d) for _ in range(in_dim)])
        for fc in self.tok:
            nn.init.kaiming_normal_(fc.weight, nonlinearity="relu"); nn.init.zeros_(fc.bias)
        self.cls = nn.Parameter(torch.randn(d) * 0.02)
        self.blocks = nn.ModuleList([TransformerBlock(d, heads, p) for _ in range(layers)])
        self.ln_f = nn.LayerNorm(d, eps=1e-5)
        self.head = nn.Linear(d, 1)
        nn.init.kaiming_normal_(self.head.weight, nonlinearity="linear"); nn.init.zeros_(self.head.bias)
    def forward(self, x):
        B = x.shape[0]
        toks = [fc(x[:, j:j+1]) for j, fc in enumerate(self.tok)]
        t = torch.stack(toks, dim=1)
        t = torch.cat([self.cls.view(1, 1, -1).expand(B, 1, -1), t], dim=1)
        for blk in self.blocks:
            t = blk(t)
        return self.head(self.ln_f(t[:, 0]))

device = torch.device("cpu")

## Grad-check (architecture-only, run once — not data-dependent)

In [4]:
_gc_model = FTTransformerLite(in_dim=len(FEATURE_COLS)).to(device)
_gc_model.eval()
xb0 = torch.from_numpy(X_full.values[:5].astype(np.float32))
xb0.requires_grad_(True)
_gc_crit = nn.BCEWithLogitsLoss()
loss0 = _gc_crit(_gc_model(xb0), torch.from_numpy(y_full[:5].astype(np.float32)).view(-1, 1))
loss0.backward()
assert torch.isfinite(xb0.grad).all(), "grad-check failed: non-finite input grad"
print(f"grad-check OK | loss {loss0.item():.4f} | input-grad mean {xb0.grad.abs().mean().item():.6f}")
del _gc_model, _gc_crit

grad-check OK | loss 1.0112 | input-grad mean 5.264838


## K-fold training (k=5) — fresh FTTransformerLite per fold, best-val-F1 checkpoint on own fold

In [5]:
K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
X_all = X_full.values
fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_full), start=1):
    Xtr_f, Xva_f = X_all[tr_idx].copy(), X_all[va_idx].copy()
    ytr_f, yva_f = y_full[tr_idx], y_full[va_idx]
    med_f = np.nanmedian(Xtr_f[:, inc_idx])
    for arr in (Xtr_f, Xva_f):
        arr[:, inc_idx] = np.where(np.isnan(arr[:, inc_idx]), med_f, arr[:, inc_idx])
    sc_f = StandardScaler().fit(Xtr_f)
    Xtr_fs, Xva_fs = sc_f.transform(Xtr_f), sc_f.transform(Xva_f)
    train_ds_f = TabularDataset(Xtr_fs, ytr_f)
    train_loader_f = DataLoader(train_ds_f, batch_size=64, shuffle=True,
                                 generator=torch.Generator().manual_seed(SEED + fold))

    torch.manual_seed(SEED + fold)
    m = FTTransformerLite(in_dim=len(FEATURE_COLS)).to(device)
    pos_f, neg_f = (ytr_f == 1).sum(), (ytr_f == 0).sum()
    crit_f = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg_f / pos_f]))
    opt_f = torch.optim.Adam(m.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)
    sch_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)

    Xv = torch.from_numpy(Xva_fs.astype(np.float32)).to(device)
    yv = torch.from_numpy(yva_f.astype(np.float32)).view(-1, 1).to(device)

    best_f1, best_state = -1.0, None
    for epoch in range(1, 101):
        m.train()
        for xb, yb in train_loader_f:
            xb, yb = xb.to(device), yb.to(device)
            opt_f.zero_grad()
            crit_f(m(xb), yb).backward()
            opt_f.step()
        sch_f.step()
        m.eval()
        with torch.no_grad():
            logits = m(Xv)
            prob_v = torch.sigmoid(logits).cpu().numpy().ravel()
            f1v = f1_score(yva_f, (prob_v >= 0.5).astype(int))
        if f1v > best_f1:
            best_f1 = f1v
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}

    m.load_state_dict(best_state)
    m.eval()
    with torch.no_grad():
        logits = m(Xv)
        loss_v = nn.BCEWithLogitsLoss()(logits, yv).item()
        prob_v = torch.sigmoid(logits).cpu().numpy().ravel()
    pred_v = (prob_v >= 0.5).astype(int)
    fold_rows.append({
        "fold": fold,
        "test_loss_BCE": round(float(loss_v), 4),
        "accuracy": round(float(accuracy_score(yva_f, pred_v)), 4),
        "precision": round(float(precision_score(yva_f, pred_v, zero_division=0)), 4),
        "recall": round(float(recall_score(yva_f, pred_v, zero_division=0)), 4),
        "f1": round(float(f1_score(yva_f, pred_v, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(yva_f, prob_v)), 4),
    })
    print(f"fold {fold}: F1={fold_rows[-1]['f1']:.4f} BCE={fold_rows[-1]['test_loss_BCE']:.4f}")

kfold_df = pd.DataFrame(fold_rows)
summary = {k: {"mean": round(float(kfold_df[k].mean()), 4), "std": round(float(kfold_df[k].std()), 4)}
           for k in ("test_loss_BCE", "accuracy", "precision", "recall", "f1", "roc_auc")}
print(kfold_df.to_string(index=False))
print(json.dumps(summary, indent=2))
with open("weights_V2_1/classification_pytorch_kfold_metrics.json", "w") as f:
    json.dump({"k": K, "folds": fold_rows, "summary": summary}, f, indent=2)

fold 1: F1=0.6400 BCE=0.3847


fold 2: F1=0.5890 BCE=0.4576


fold 3: F1=0.5514 BCE=0.4094


fold 4: F1=0.5568 BCE=0.4081


fold 5: F1=0.5854 BCE=0.3795
 fold  test_loss_BCE  accuracy  precision  recall     f1  roc_auc
    1         0.3847    0.8795     0.5783  0.7164 0.6400   0.8886
    2         0.4576    0.8501     0.5000  0.7164 0.5890   0.8544
    3         0.4094    0.8143     0.4322  0.7612 0.5514   0.8664
    4         0.4081    0.8255     0.4495  0.7313 0.5568   0.8448
    5         0.3795    0.8479     0.4898  0.7273 0.5854   0.8691
{
  "test_loss_BCE": {
    "mean": 0.4079,
    "std": 0.0309
  },
  "accuracy": {
    "mean": 0.8435,
    "std": 0.0252
  },
  "precision": {
    "mean": 0.49,
    "std": 0.0567
  },
  "recall": {
    "mean": 0.7305,
    "std": 0.0184
  },
  "f1": {
    "mean": 0.5845,
    "std": 0.0352
  },
  "roc_auc": {
    "mean": 0.8647,
    "std": 0.0165
  }
}
